<a href="https://colab.research.google.com/github/fauzyy2948-del/UTS-BIG_DATA-AHMAD_FAUZY-14022300036/blob/main/UTS-BIG_DATA-AHMAD_FAUZY-14022300036.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install google_play_scraper

In [ ]:
from google_play_scraper import reviews, Sort
import csv

result, _ = reviews(
    'com.kai.kaiticketing',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=100,
    filter_score_with=None
)

filename = 'ulasan_google_play.csv'


with open(filename, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['userName', 'score', 'at', 'content'])
    writer.writeheader()
    for review in result:

        writer.writerow({
            'userName': review['userName'],
            'score': review['score'],
            'at': review['at'],
            'content': review['content']
        })

print(f"Berhasil menyimpan {len(result)} ulasan ke '{filename}'")

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

First, let's load the saved reviews from the CSV file into a pandas DataFrame.

In [ ]:
df = pd.read_csv('ulasan_google_play.csv')
display(df.head())

Now, let's load the sentiment analysis model and its tokenizer from Hugging Face. We'll use the `w11wo/indonesian-roberta-base-sentiment-classifier` model as requested.

In [ ]:
model_name = 'w11wo/indonesian-roberta-base-sentiment-classifier'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Check if GPU is available and move the model to GPU if it is
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"Using device: {device}")

Next, we'll define a function to predict the sentiment of a given text using the loaded model. This function will return the sentiment label (e.g., 'positive', 'negative', 'neutral') and its corresponding probability score.

In [ ]:
def get_sentiment(text):
    if pd.isna(text) or not isinstance(text, str) or not text.strip():
        return None, None # Handle empty or NaN content
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    # Move inputs to the same device as the model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    # Get probabilities from logits and apply softmax
    probabilities = torch.softmax(outputs.logits, dim=1)

    # Get the predicted class index
    predicted_class_idx = torch.argmax(probabilities, dim=1).item()

    # Get the label from the model's config
    predicted_label = model.config.id2label[predicted_class_idx]
    predicted_score = probabilities[0, predicted_class_idx].item()

    return predicted_label, predicted_score

Now, let's apply this sentiment analysis function to the 'content' column of our DataFrame. This might take a few moments depending on the number of reviews.

In [ ]:
# Apply sentiment analysis to each review content
df[['sentiment', 'sentiment_score']] = df['content'].apply(lambda x: pd.Series(get_sentiment(x)))

display(df.head())

Finally, let's look at the distribution of sentiments in your reviews.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sentiment_counts = df['sentiment'].value_counts(dropna=False)

plt.figure(figsize=(8, 6))
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values, palette='viridis')
plt.title('Distribution of Sentiments in Google Play Reviews')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.show()

print("Sentiment Distribution:")
print(sentiment_counts)